In [ ]:
#LEAKY BUCKET PROGRAM
!apt-get update -qq
!apt-get install -y gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:13.2.0-7ubuntu1).
gcc set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 48 not upgraded.


In [ ]:
%%writefile leaky.c
#include <stdio.h>

int min(int x, int y)
{
    if (x < y)
        return x;
    else
        return y;
}

int main()
{
    int drop = 0, mini, nsec, cap;
    int count = 0, i, inp[25], process;

    printf("Enter the bucket size:\n");
    scanf("%d", &cap);

    printf("Enter the processing rate:\n");
    scanf("%d", &process);

    printf("Enter the number of seconds you want to simulate:\n");
    scanf("%d", &nsec);

    for (i = 0; i < nsec; i++)
    {
        printf("Enter the size of the packet entering at %d sec:\n", i + 1);
        scanf("%d", &inp[i]);
    }

    printf("\nSecond | Packet Received | Packet Sent | Packet Left | Dropped\n");
    printf("--------------------------------------------------------------\n");

    for (i = 0; i < nsec; i++)
    {
        count += inp[i];

        if (count > cap)
        {
            drop = count - cap;
            count = cap;
        }

        printf("%d\t %d\t\t", i + 1, inp[i]);

        mini = min(count, process);
        printf("%d\t\t", mini);

        count -= mini;

        printf("%d\t\t %d\n", count, drop);
        drop = 0;
    }

    for (; count != 0; i++)
    {
        printf("%d\t 0\t\t", i + 1);

        mini = min(count, process);
        printf("%d\t\t", mini);

        count -= mini;

        printf("%d\t\t %d\n", count, drop);
        drop = 0;
    }

    return 0;
}

Overwriting leaky.c


In [ ]:
!gcc leaky.c -o leaky
!./leaky

Enter the bucket size:
5
Enter the processing rate:
3
Enter the number of seconds you want to simulate:
2
Enter the size of the packet entering at 1 sec:
7
Enter the size of the packet entering at 2 sec:
4

Second | Packet Received | Packet Sent | Packet Left | Dropped
--------------------------------------------------------------
1	 7		3		2		 2
2	 4		3		2		 1
3	 0		2		0		 0


In [ ]:
#tcp/ip program
# Create a sample file

with open("sample.txt", "w") as f:
    f.write("Hello! This is a TCP socket programming demonstration.\n")
    f.write("This file is sent from the server to the client.\n")
    f.write("TCP communication is successful!")

print("File created successfully!")


File created successfully!


In [ ]:
#server program
import socket
import os
import threading

HOST = '127.0.0.1'
PORT = 65432

server_ready = threading.Event()

def run_server():

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:

        server_socket.bind((HOST, PORT))

        server_socket.listen()

        print("Server listening on {}:{}".format(HOST, PORT))

        server_ready.set()

        conn, addr = server_socket.accept()

        with conn:

            print("Connected by", addr)

            filename = conn.recv(1024).decode()

            print("Requested file:", filename)

            if os.path.isfile(filename):

                with open(filename, 'rb') as f:
                    content = f.read()

                conn.sendall(content)

                print("File sent successfully!")

            else:
                conn.sendall(b"File not found.")

            conn.shutdown(socket.SHUT_WR)


# Start server in background
server_thread = threading.Thread(target=run_server, daemon=True)

server_thread.start()

server_ready.wait()

print("Server is ready for client connection!")


Server listening on 127.0.0.1:65432
Server is ready for client connection!


In [ ]:
#client server
import socket

HOST = '127.0.0.1'
PORT = 65432

filename = input("Enter file name to request: ")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:

    client_socket.connect((HOST, PORT))

    client_socket.sendall(filename.encode())

    data = b''

    while True:
        chunk = client_socket.recv(4096)

        if not chunk:
            break

        data += chunk

    print("\n--- File Content ---")
    print(data.decode())


Enter file name to request: sample.txt
Connected by ('127.0.0.1', 46114)
Requested file: sample.txt
File sent successfully!

--- File Content ---
Hello! This is a TCP socket programming demonstration.
This file is sent from the server to the client.
TCP communication is successful!


In [ ]:
#udp sockets

# Create a sample file

with open("sample.txt", "w") as f:
    f.write("Hello! This is a UDP socket programming demonstration.\n")
    f.write("The server is sending this file to the client.\n")
    f.write("UDP communication is successful!")

print("File created successfully!")


File created successfully!


In [ ]:
#udp server
import socket
import os
import threading

HOST = '127.0.0.1'
PORT = 5005

server_ready = threading.Event()

def run_server():

    # Create UDP socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

    server_socket.bind((HOST, PORT))

    print(f"UDP Server listening on {HOST}:{PORT}")

    server_ready.set()

    while True:

        filename, client_addr = server_socket.recvfrom(1024)

        filename = filename.decode()

        print(f"Client requested file: {filename}")

        if os.path.isfile(filename):

            with open(filename, 'r') as f:
                data = f.read()

            server_socket.sendto(data.encode(), client_addr)

            print("File sent successfully!")

        else:

            server_socket.sendto(
                b"ERROR: File not found",
                client_addr
            )

            print("File not found!")


# Start server in background
server_thread = threading.Thread(target=run_server, daemon=True)

server_thread.start()

server_ready.wait()

print("UDP Server is ready!")


UDP Server listening on 127.0.0.1:5005
UDP Server is ready!


In [ ]:
#udp client

import socket

HOST = '127.0.0.1'
PORT = 5005

# Create UDP socket
client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

filename = input("Enter filename to request: ")

# Send filename to server
client_socket.sendto(filename.encode(), (HOST, PORT))

# Receive file contents
data, server_addr = client_socket.recvfrom(4096)

print("\n--- File Content ---")
print(data.decode())

client_socket.close()


Enter filename to request: sample.txt
Client requested file: sample.txt
File sent successfully!

--- File Content ---
Hello! This is a UDP socket programming demonstration.
The server is sending this file to the client.
UDP communication is successful!
